In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🛠️ Product Documentation Copilot

## What We're Building

A chat assistant that can answer questions over **API docs**, **code examples**,
and **changelogs** — with **structured citations** showing exactly where each
piece of information came from.

## The Problem

Developers spend hours reading docs. A docs copilot lets them:
- "How do I authenticate?" → Link to auth docs + code example
- "What changed in v2.3?" → Relevant changelog entries
- "Show me a pagination example" → Working code from examples

## Key Design Decision: Document Categories

We tag every chunk with its **category** (api_reference, example, changelog)
so the LLM knows whether it's looking at a spec, sample code, or update notes.

## Stack
- **LangChain** — retrieval + generation
- **ChromaDB** — vector store with category metadata
- **Ollama** — local LLM + embeddings
- **Structured citations** — formatted source references

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter, Language
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Product Documentation

We simulate docs for a fictional API called **"DataStream API"** — a real-time
data streaming service. Three doc types: API reference, code examples, changelog.

In [ ]:
api_reference = """# DataStream API Reference

## Authentication

All API requests require a Bearer token in the Authorization header.

```
Authorization: Bearer <your-api-key>
```

API keys are managed at https://dashboard.datastream.io/keys.
Keys can have scoped permissions: read, write, admin.
Rate limits: 1000 requests/minute for read, 100/minute for write.

## POST /streams

Create a new data stream.

Request Body:
- name (string, required): Stream name, 3-64 characters
- schema (object, required): JSON Schema for stream events
- retention_days (integer, optional): Data retention period, default 30
- partitions (integer, optional): Number of partitions, default 4

Response: 201 Created
- id (string): Unique stream identifier
- name (string): Stream name
- status (string): "active" or "provisioning"
- created_at (string): ISO 8601 timestamp

Error Codes:
- 400: Invalid schema or name
- 409: Stream name already exists
- 429: Rate limit exceeded

## GET /streams/{stream_id}/events

Read events from a stream.

Query Parameters:
- limit (integer, optional): Max events to return, default 100, max 1000
- offset (string, optional): Cursor for pagination
- start_time (string, optional): ISO 8601 filter start
- end_time (string, optional): ISO 8601 filter end

Response: 200 OK
- events (array): List of event objects
- next_offset (string|null): Cursor for next page
- total_count (integer): Total matching events

## POST /streams/{stream_id}/events

Publish events to a stream.

Request Body:
- events (array, required): Array of event objects matching stream schema
- batch_id (string, optional): Idempotency key for the batch

Response: 202 Accepted
- accepted (integer): Number of events accepted
- rejected (integer): Number of events rejected due to schema violations
- errors (array): Details of rejected events

## DELETE /streams/{stream_id}

Delete a stream and all its data. This action is irreversible.

Response: 204 No Content
Error: 404 if stream not found.

## Webhooks

Configure webhooks to receive real-time notifications.

POST /webhooks
- url (string, required): HTTPS endpoint to receive events
- stream_id (string, required): Stream to subscribe to
- events (array, optional): Event types to filter, default all
- secret (string, optional): Signing secret for HMAC verification

Webhook payloads include an X-Signature header for verification.
"""


code_examples = """# DataStream API — Code Examples

## Example 1: Create a Stream and Publish Events (Python)

```python
import requests

API_KEY = "your-api-key"
BASE_URL = "https://api.datastream.io/v2"
headers = {"Authorization": f"Bearer {API_KEY}"}

# Create stream
stream = requests.post(f"{BASE_URL}/streams", headers=headers, json={
    "name": "user-events",
    "schema": {
        "type": "object",
        "properties": {
            "user_id": {"type": "string"},
            "action": {"type": "string"},
            "timestamp": {"type": "string", "format": "date-time"}
        },
        "required": ["user_id", "action"]
    },
    "retention_days": 90
}).json()

stream_id = stream["id"]
print(f"Created stream: {stream_id}")

# Publish events
result = requests.post(f"{BASE_URL}/streams/{stream_id}/events",
    headers=headers,
    json={"events": [
        {"user_id": "u123", "action": "login"},
        {"user_id": "u456", "action": "purchase"},
    ]}
).json()
print(f"Accepted: {result['accepted']}, Rejected: {result['rejected']}")
```

## Example 2: Paginated Reading (Python)

```python
def read_all_events(stream_id, start="2024-01-01T00:00:00Z"):
    all_events = []
    offset = None
    while True:
        params = {"limit": 1000, "start_time": start}
        if offset:
            params["offset"] = offset
        resp = requests.get(
            f"{BASE_URL}/streams/{stream_id}/events",
            headers=headers, params=params
        ).json()
        all_events.extend(resp["events"])
        offset = resp.get("next_offset")
        if not offset:
            break
    return all_events
```

## Example 3: Webhook with Signature Verification (Python)

```python
import hmac, hashlib

WEBHOOK_SECRET = "your-webhook-secret"

def verify_webhook(payload_bytes, signature):
    expected = hmac.new(
        WEBHOOK_SECRET.encode(), payload_bytes, hashlib.sha256
    ).hexdigest()
    return hmac.compare_digest(expected, signature)

# In your webhook handler:
# signature = request.headers["X-Signature"]
# if not verify_webhook(request.body, signature):
#     return 401
```

## Example 4: Error Handling Best Practices

```python
import time

def publish_with_retry(stream_id, events, max_retries=3):
    for attempt in range(max_retries):
        resp = requests.post(
            f"{BASE_URL}/streams/{stream_id}/events",
            headers=headers, json={"events": events}
        )
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt))
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception("Max retries exceeded")
```
"""


changelog = """# DataStream API — Changelog

## v2.5.0 (2024-12-15)
- NEW: Added `batch_id` field to POST /events for idempotent writes
- NEW: Webhook signature verification using HMAC-SHA256
- IMPROVED: Pagination now uses cursor-based offsets (breaking change from page numbers)
- FIX: Fixed race condition in concurrent stream creation
- DEPRECATION: `page` and `per_page` query params removed from GET /events

## v2.4.0 (2024-09-01)
- NEW: Added DELETE /streams endpoint
- NEW: Stream partitioning support (1-64 partitions)
- IMPROVED: Rate limits increased from 500 to 1000 req/min for read operations
- FIX: Fixed incorrect `total_count` when using time range filters
- SECURITY: API keys now support scoped permissions (read/write/admin)

## v2.3.0 (2024-06-01)
- NEW: Time range filtering with `start_time` and `end_time` parameters
- NEW: Webhook event type filtering
- IMPROVED: Schema validation now returns detailed error messages per event
- FIX: Fixed webhook delivery retries not respecting exponential backoff
- PERFORMANCE: Event ingestion throughput improved by 40%

## v2.2.0 (2024-03-01)
- NEW: Event schema validation on publish
- NEW: `retention_days` configuration per stream
- IMPROVED: Dashboard now shows real-time stream metrics
- FIX: Fixed timezone handling in event timestamps

## v2.1.0 (2023-12-01)
- NEW: Webhooks support for real-time event delivery
- NEW: Admin API for managing API keys programmatically
- IMPROVED: Response times reduced by 30% through query optimization

## v2.0.0 (2023-09-01) — MAJOR RELEASE
- BREAKING: New authentication system (API keys replace OAuth tokens)
- BREAKING: Event format changed to JSON Schema validation
- NEW: Multi-tenant stream isolation
- NEW: Cross-region replication
- IMPROVED: Complete API redesign for consistency
"""

print("Loaded 3 documentation sources:")
print(f"  API Reference: {len(api_reference.split())} words")
print(f"  Code Examples: {len(code_examples.split())} words")
print(f"  Changelog:     {len(changelog.split())} words")

## Step 3 — Index with Category Metadata

Each chunk is tagged with its **doc_type** so the copilot knows whether
it's referencing a spec, a code example, or a changelog entry.

In [ ]:
def index_docs(
    text: str,
    doc_type: str,
    source_file: str,
) -> list[Document]:
    """Chunk a doc and tag every chunk with its category."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=80,
        separators=["\n## ", "\n\n", "\n", ". "],
    )
    base_doc = Document(
        page_content=text,
        metadata={
            "doc_type": doc_type,
            "source": source_file,
        },
    )
    return splitter.split_documents([base_doc])


all_chunks = []
all_chunks += index_docs(api_reference, "api_reference", "api-reference.md")
all_chunks += index_docs(code_examples, "code_example", "examples.md")
all_chunks += index_docs(changelog, "changelog", "CHANGELOG.md")

print(f"Total chunks: {len(all_chunks)}")
from collections import Counter
counts = Counter(c.metadata["doc_type"] for c in all_chunks)
for dtype, count in counts.items():
    print(f"  {dtype}: {count}")

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="docs_copilot",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print(f"Indexed {vectorstore._collection.count()} doc chunks")

## Step 4 — Copilot with Structured Citations

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

copilot_prompt = ChatPromptTemplate.from_template(
    """You are a product documentation copilot for the DataStream API.
Answer developer questions using the retrieved documentation.

GUIDELINES:
- Include code snippets when relevant (copy from examples, don't invent)
- Cite the source type: [API Ref], [Example], or [Changelog]
- For "what changed" questions, reference specific version numbers
- For "how to" questions, prefer showing a code example
- If the answer involves a breaking change, warn prominently

Documentation Context:
{context}

Developer Question: {question}

Answer:"""
)


def ask_copilot(question: str) -> None:
    """Ask the docs copilot a question."""
    print(f"💬 {question}")
    print("=" * 60)
    
    docs = retriever.invoke(question)
    
    # Show sources
    print("\n📋 Sources retrieved:")
    for i, doc in enumerate(docs, 1):
        dtype = doc.metadata['doc_type']
        src = doc.metadata['source']
        preview = doc.page_content[:50].replace('\n', ' ').strip()
        print(f"  [{i}] ({dtype}) {src} — \"{preview}...\"")
    print()
    
    # Build context with source labels
    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata['source']} | Type: {d.metadata['doc_type']}]\n"
        f"{d.page_content}"
        for d in docs
    )
    
    chain = copilot_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    print(answer)
    print("\n" + "=" * 60)


print("Docs copilot ready!")

## Step 5 — Ask Dev Questions

In [ ]:
ask_copilot("How do I authenticate with the API?")

In [ ]:
ask_copilot("Show me how to paginate through events in Python")

In [ ]:
ask_copilot("What changed in version 2.5? Are there any breaking changes?")

In [ ]:
ask_copilot("How do I verify webhook signatures?")

In [ ]:
ask_copilot("What are the rate limits and how should I handle 429 errors?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Doc type categorization** | Tags chunks as api_reference, code_example, or changelog |
| **Source-aware retrieval** | Shows which doc file each chunk came from |
| **Structured citations** | [API Ref], [Example], [Changelog] labels in answers |
| **Code-preserving chunks** | Section-aware splitting keeps code blocks intact |
| **Version-aware answers** | Changelog context enables "what changed" queries |

## 🔧 Customization Ideas

- **Multi-version docs**: Index docs for v2.3, v2.4, v2.5 separately
- **Interactive playground**: Let users run code examples directly
- **Markdown rendering**: Format responses as proper markdown with syntax highlighting
- **Feedback loop**: Track which questions users ask most to improve docs